In [1]:
# Проверка кода по сохранению отчетности в xlsx
import re
import requests
from pathlib import Path
import zipfile, pandas as pd


ORG_ID = 1454655                     # ID со страницы organizations-card
OUT_DIR = Path("test_bo_nalog_reports")   # папка для сохранения архивов
OUT_DIR.mkdir(exist_ok=True)
BASE = "https://bo.nalog.gov.ru"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ru,en;q=0.9",
    "Referer": f"{BASE}/organizations-card/{ORG_ID}",
}

session = requests.Session()
session.headers.update(HEADERS)

# Прогреваем сессию (устанавливаем куки) - заходим на карточку организации
session.get(f"{BASE}/organizations-card/{ORG_ID}", timeout=30)

# Получаем список отчётов (БФО) организации
list_url = f"{BASE}/nbo/organizations/{ORG_ID}/bfo/"
r = session.get(list_url, timeout=30)
r.raise_for_status()
reports = r.json()
print(f"Найдено отчётов: {len(reports)}")

for rep in reports:
    period      = rep.get("period")
    target_corr = rep.get("publishedCorrectionNumber")

    print("Таргетная корректировка ", target_corr)

    detail_id = rep.get("typeCorrections")[0].get("correction").get("id")

    #print(rep.get("typeCorrections")[0].get("correction").get("id"))

    if detail_id is None:
        print(f"  [skip] {period}: не найдена корректировка №{target_corr}")
        continue

    zip_url = f"https://bo.nalog.gov.ru/download/bfo/{ORG_ID}?auditReport=false&balance=true&capitalChange=true&clarification=false&targetedFundsUsing=false&detailsId={detail_id}&financialResult=true&fundsMovement=true&type=XLS&period={period}"
    resp = session.get(zip_url, timeout=60, stream=True)

    if resp.status_code != 200:
        print(f"  [skip] {period} (id={detail_id}) HTTP {resp.status_code}")
        continue

    # имя файла: год + (корректировка, если не первичная)
    #suffix = "" if not corr else f"_corr{corr}"
    fname  = OUT_DIR / f"bfo_{ORG_ID}_{period}_{target_corr}_id{detail_id}.zip"

    with open(fname, "wb") as f:
        for chunk in resp.iter_content(chunk_size=64 * 1024):
            if chunk:
                f.write(chunk)
    print(f"{period}  {detail_id}: {fname.name} ({fname.stat().st_size/1024:.1f} KB)")

print(f"\nАрхивы сохранены в: {OUT_DIR.resolve()}")

for zp in OUT_DIR.glob("*.zip"):
    with zipfile.ZipFile(zp) as z:
        z.extractall(zp.with_suffix(""))

Найдено отчётов: 5
Таргетная корректировка  0
2025  57009257: bfo_1454655_2025_0_id57009257.zip (35.8 KB)
Таргетная корректировка  0
2024  52020316: bfo_1454655_2024_0_id52020316.zip (41.2 KB)
Таргетная корректировка  0
2023  48441953: bfo_1454655_2023_0_id48441953.zip (41.1 KB)
Таргетная корректировка  0
2022  45783500: bfo_1454655_2022_0_id45783500.zip (41.6 KB)
Таргетная корректировка  1
2021  42762517: bfo_1454655_2021_1_id42762517.zip (41.4 KB)

Архивы сохранены в: C:\Users\GaV\Desktop\Parsers\test_bo_nalog_reports


In [2]:
# Пути и настройки
import re
import html
import shutil
import zipfile
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import pandas as pd
import requests
from openpyxl import load_workbook

INPUT_XLSX = r"C:\Users\GaV\Desktop\Parsers\output_list_files.xlsx"
OUTPUT_XLSX = r"C:\Users\GaV\Desktop\Parsers\output_list_files.xlsx"
COMPANIES_XLSX = r"C:\Users\GaV\Desktop\Parsers\Companies.xlsx"

BASE_SAVE_DIR = Path(r"C:\FTSFiles")
BASE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

MAX_ROWS_TO_PROCESS = 10000

BASE = "https://bo.nalog.gov.ru"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ru,en;q=0.9",
}

session = requests.Session()
session.headers.update(HEADERS)

In [3]:
# Вспомогательные функции

def is_filled(value) -> bool:
    if pd.isna(value):
        return False
    s = str(value).strip()
    return s != "" and s.lower() != "nan"


def clean_inn(value) -> str:
    if not is_filled(value):
        return ""
    return re.sub(r"\D", "", str(value))


def sanitize_filename(text: str) -> str:
    text = str(text).strip()
    text = text.replace('"', "_")
    text = re.sub(r'[<>:"/\\|?*«»]', "_", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def print_red(*args, **kwargs):
    RED = "\033[31m"
    RESET = "\033[0m"
    text = " ".join(str(a) for a in args)
    print(f"{RED}{text}{RESET}", **kwargs)

In [4]:
# Загрузка Excel-файлов

df = pd.read_excel(INPUT_XLSX)
companies_df = pd.read_excel(COMPANIES_XLSX)

# В output_list_files ожидаем эти колонки
required_df_cols = ["Тип документа", "Отчетный период", "FileID", "company_name"]
missing_df_cols = [c for c in required_df_cols if c not in df.columns]
if missing_df_cols:
    raise ValueError(f"В {INPUT_XLSX} отсутствуют обязательные колонки: {missing_df_cols}")

if "ИНН" not in df.columns:
    df["ИНН"] = pd.NA

required_companies_cols = ["Наименование", "Код ФНС"]
missing_companies_cols = [c for c in required_companies_cols if c not in companies_df.columns]
if missing_companies_cols:
    raise ValueError(f"В {COMPANIES_XLSX} отсутствуют обязательные колонки: {missing_companies_cols}")

for col in ["ОФП ФНС", "ОФР ФНС", "ОИК ФНС", "ОДДС ФНС"]:
    if col not in df.columns:
        df[col] = ""

In [5]:
# Поиск Код ФНС / ORG_ID в Companies.xlsx
# если в output_list_files заполнен ИНН, ищем по ИНН
# если ИНН пустой, ищем по точному company_name == Наименование без нормализации


def get_fns_org_id_from_companies(row: pd.Series, companies_df: pd.DataFrame) -> Optional[int]:
    row_inn = clean_inn(row.get("ИНН"))
    row_company_name = row.get("company_name")

    # 1. Поиск по ИНН
    if row_inn:
        if len(row_inn) == 10 and row_inn.isdigit():
            companies_inn = companies_df["ИНН"].apply(clean_inn)
            match = companies_df.loc[companies_inn == row_inn]

            if match.empty:
                print(f"Не найден Код ФНС по ИНН={row_inn} для company_name={row_company_name!r}")
                return None

            code_fns = match.iloc[0]["Код ФНС"]
            if pd.isna(code_fns):
                print(f"Для ИНН={row_inn} найденная строка не содержит Код ФНС")
                return None

            return int(code_fns)
    else:
        print(f"ИНН имеет некорректный формат: {row_inn!r}, поиск по названию")
    # 2. Точное совпадение по company_name == Наименование
    if not is_filled(row_company_name):
        print("Пустые и ИНН, и company_name — Код ФНС определить нельзя")
        return None

    match = companies_df.loc[companies_df["Наименование"] == row_company_name]

    if match.empty:
        print(f"Не найден Код ФНС по точному названию company_name={row_company_name!r}")
        return None

    if len(match) > 1:
        print(f"Найдено несколько совпадений по названию {row_company_name!r}, берется первая строка")

    code_fns = match.iloc[0]["Код ФНС"]
    if pd.isna(code_fns):
        print(f"Для company_name={row_company_name!r} найденная строка не содержит Код ФНС")
        return None

    return int(code_fns)

In [6]:
# Получение detail_id и скачивание zip-архива БФО

def get_detail_id_for_period(orgid: int, period: int, session: requests.Session) -> Optional[int]:
    session.get(f"{BASE}/organizations-card/{orgid}", timeout=30)

    list_url = f"{BASE}/nbo/organizations/{orgid}/bfo/"
    r = session.get(list_url, timeout=30)
    r.raise_for_status()
    reports = r.json()
    
    candidates = []

    for rep in reports:
        rep_period = rep.get("period")
        if rep_period != str(period):
            continue

        target_corr = rep.get("publishedCorrectionNumber")
        type_corrections = rep.get("typeCorrections") or []

        if not type_corrections:
            continue

        correction = type_corrections[0].get("correction") or {}

        detail_id = correction.get("id")
        if detail_id is not None:
            candidates.append((target_corr, detail_id))

    if not candidates:
        return None

    candidates = sorted(candidates, key=lambda x: (x[0] is None, x[0]))
    
    return candidates[0][1]


def download_bfo_zip(orgid: int, period: int, detail_id: int, out_zip_path: Path, session: requests.Session) -> Path:
    zip_url = (
        f"{BASE}/download/bfo/{orgid}"
        f"?auditReport=false"
        f"&balance=true"
        f"&capitalChange=true"
        f"&clarification=false"
        f"&targetedFundsUsing=false"
        f"&detailsId={detail_id}"
        f"&financialResult=true"
        f"&fundsMovement=true"
        f"&type=XLS"
        f"&period={period}"
    )

    resp = session.get(zip_url, timeout=120, stream=True)
    resp.raise_for_status()

    out_zip_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_zip_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=64 * 1024):
            if chunk:
                f.write(chunk)

    return out_zip_path


def extract_zip(zip_path: Path, extract_dir: Path) -> Path:
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    return extract_dir


def find_first_xlsx(folder: Path) -> Optional[Path]:
    xlsx_files = list(folder.rglob("*.xlsx"))
    return xlsx_files[0] if xlsx_files else None

In [7]:
# настройки распознавания листов

SHEET_PATTERNS: dict[str, list[str]] = {
    "balance": ["бухгалтерский баланс", "баланс", "офп","финаносовом положении"],
    "pnl": ["финансовых результатах", "прибылях и убытках", "офр"],
    "equity": ["изменениях капитала", "изменении капитала", "оик"],
    "cashflow": ["движении денежных средств", "движении денежных", "оддс"],
}

HEADER_KEYWORDS = {
    "name_col": ["наименование показателя", "показатель", "пояснения","примечания"],
    "code_col": ["код строки", "код строк", "код"],
}

HEADER_STOPWORDS = ["инн", "кпп", "кнд", "окуд", "форма по", "тыс.", "млн", "руб."]
FOOTNOTE_KEYWORD = "примеч"

SECTION_STOPWORDS = ["корректировки в связи с изменением учетной политики", "исправлением ошибок", "3. чистые активы"]


@dataclass
class Issue:
    code: str
    level: str
    message: str


@dataclass
class SheetResult:
    sheet_name: str
    report_type: Optional[str]
    status: str
    issues: list = field(default_factory=list)
    title: Optional[str] = None
    date: Optional[str] = None
    html: Optional[str] = None


@dataclass
class FileResult:
    file_path: str
    sheets: list = field(default_factory=list)
    missing_report_types: list = field(default_factory=list)

    @property
    def overall_status(self) -> str:
        if any(s.status == "ERROR" for s in self.sheets):
            return "ERROR"
        if any(s.status == "WARN" for s in self.sheets):
            return "WARN"
        return "OK"

In [8]:
# разбор xlsx-листов и html-рендеринг

def _has_border(cell) -> bool:
    b = cell.border
    if b is None:
        return False
    return any(
        getattr(b, s).border_style
        for s in ("top", "bottom", "left", "right")
        if getattr(b, s) is not None
    )


def _cell_texts(ws, row_idx: int) -> list:
    return [
        str(cell.value).strip()
        for cell in ws[row_idx]
        if cell.value not in (None, "")
    ]


def _find_header_row(ws, search_limit: int = 20) -> Optional[int]:
    for ri in range(1, min(search_limit + 1, ws.max_row + 1)):
        texts_lower = [t.lower() for t in _cell_texts(ws, ri)]
        has_name = any(kw in t for t in texts_lower for kw in HEADER_KEYWORDS["name_col"])
        has_code = any(kw in t for t in texts_lower for kw in HEADER_KEYWORDS["code_col"])
        if has_name and has_code:
            return ri
    return None


def _find_title_date(ws, header_row: int) -> tuple[Optional[str], Optional[str]]:
    candidates = []
    for ri in range(1, header_row):
        if any(_has_border(c) for c in ws[ri]):
            continue
        texts = _cell_texts(ws, ri)
        if not texts:
            continue
        combined_lower = " ".join(texts).lower()
        if any(sw in combined_lower for sw in HEADER_STOPWORDS):
            continue
        candidates.append(" ".join(texts))

    title = candidates[-2] if len(candidates) >= 2 else (candidates[-1] if candidates else None)
    date = candidates[-1] if len(candidates) >= 2 else None
    return title, date


def _find_table_end(ws, header_row: int) -> Optional[int]:
    stop_row = None

    for ri in range(header_row + 1, ws.max_row + 1):

        texts = _cell_texts(ws, ri)
        if not texts:
            continue
        combined_lower = " ".join(texts).lower()
        #print(combined_lower)

        if any(sw in combined_lower for sw in SECTION_STOPWORDS):
            stop_row = ri
            break

        if FOOTNOTE_KEYWORD in combined_lower:
            stop_row = ri
            break

    search_until = stop_row if stop_row else ws.max_row + 1
    last = None
    for ri in range(header_row + 1, search_until):
        has_brd = any(_has_border(c) for c in ws[ri])
        has_dat = any(c.value not in (None, "") for c in ws[ri])
        if has_brd and has_dat:
            last = ri
    return last

def _find_col_bounds(ws, header_row: int) -> tuple[int, int]:
    cols = [cell.column for cell in ws[header_row] if cell.value not in (None, "")]
    return (min(cols), max(cols)) if cols else (1, ws.max_column)


def _build_merged_map(ws) -> dict:
    merged_map = {}
    for rng in ws.merged_cells.ranges:
        info = (rng.min_row, rng.min_col, rng.max_row, rng.max_col)
        for ri in range(rng.min_row, rng.max_row + 1):
            for ci in range(rng.min_col, rng.max_col + 1):
                merged_map[(ri, ci)] = info
    return merged_map


def render_html_table(ws, header_row: int, table_end: int, min_col: int, max_col: int) -> str:
    merged_map = _build_merged_map(ws)
    skipped = set()
    rows_html = []

    for ri in range(header_row, table_end + 1):
        cells_html = []
        for ci in range(min_col, max_col + 1):
            if (ri, ci) in skipped:
                continue

            tag = "th" if ri == header_row else "td"
            raw_val = ws.cell(row=ri, column=ci).value
            val = html.escape(str(raw_val or "").strip())

            colspan = rowspan = 1
            if (ri, ci) in merged_map:
                mr0, mc0, mr1, mc1 = merged_map[(ri, ci)]
                if mr0 == ri and mc0 == ci:
                    colspan = mc1 - mc0 + 1
                    rowspan = mr1 - mr0 + 1
                    for rr in range(mr0, mr1 + 1):
                        for cc in range(mc0, mc1 + 1):
                            if (rr, cc) != (ri, ci):
                                skipped.add((rr, cc))
                else:
                    continue

            attrs = ""
            if colspan > 1:
                attrs += f' colspan="{colspan}"'
            if rowspan > 1:
                attrs += f' rowspan="{rowspan}"'

            cells_html.append(f"<{tag}{attrs}>{val}</{tag}>")

        if cells_html:
            rows_html.append("<tr>" + "".join(cells_html) + "</tr>")

    return '<table border="1" cellspacing="0" cellpadding="4">' + "".join(rows_html) + "</table>"


def detect_report_type(sheet_name: str) -> Optional[str]:
    name_lower = sheet_name.lower().strip()
    for report_type, patterns in SHEET_PATTERNS.items():
        if any(p in name_lower for p in patterns):
            return report_type
    return None


def process_sheet(ws, sheet_name: str) -> SheetResult:
    issues = []
    report_type = detect_report_type(sheet_name)

    header_row = _find_header_row(ws)
    if header_row is None:
        issues.append(Issue("HEADERNOTFOUND", "ERROR", "Не найдена строка заголовка таблицы"))
        return SheetResult(sheet_name, report_type, "ERROR", issues)

    table_end = _find_table_end(ws, header_row)
    if table_end is None or table_end <= header_row:
        issues.append(Issue("EMPTYTABLE", "ERROR", "После заголовка не найдена таблица"))
        return SheetResult(sheet_name, report_type, "ERROR", issues)

    min_col, max_col = _find_col_bounds(ws, header_row)
    title, date = _find_title_date(ws, header_row)

    if title is None:
        issues.append(Issue("TITLENOTFOUND", "WARN", "Не найден заголовок формы"))
    if date is None:
        issues.append(Issue("DATENOTFOUND", "WARN", "Не найдена дата формы"))
    if report_type is None:
        issues.append(Issue("UNKNOWNSHEETTYPE", "WARN", "Не определен тип листа"))

    try:
        html_table = render_html_table(ws, header_row, table_end, min_col, max_col)
    except Exception as exc:
        issues.append(Issue("HTMLRENDERERROR", "ERROR", f"Ошибка при построении HTML: {exc}"))
        return SheetResult(sheet_name, report_type, "ERROR", issues, title=title, date=date)

    status = "OK" if not issues else ("ERROR" if any(i.level == "ERROR" for i in issues) else "WARN")
    return SheetResult(sheet_name, report_type, status, issues, title=title, date=date, html=html_table)


def process_file(filepath: str, target_sheet_names: Optional[list] = None) -> FileResult:
    result = FileResult(file_path=filepath)

    try:
        wb = load_workbook(filepath, data_only=True)
    except Exception as exc:
        result.sheets.append(
            SheetResult(
                sheet_name="file",
                report_type=None,
                status="ERROR",
                issues=[Issue("FILEOPENERROR", "ERROR", str(exc))]
            )
        )
        return result

    sheets_to_process = [
        name for name in wb.sheetnames
        if (
            (target_sheet_names is None and detect_report_type(name) is not None)
            or (target_sheet_names is not None and name in target_sheet_names)
        )
    ]

    found_types = [detect_report_type(n) for n in sheets_to_process]
    result.missing_report_types = [rt for rt in SHEET_PATTERNS if rt not in found_types]

    for sheet_name in sheets_to_process:
        ws = wb[sheet_name]
        result.sheets.append(process_sheet(ws, sheet_name))

    return result

In [9]:
# Сохранение html-форм

REPORTTYPE_TO_FILENAME = {
    "balance": "ОФП.html",
    "pnl": "ОФР.html",
    "equity": "ОИК.html",
    "cashflow": "ОДДС.html",
}

REPORTTYPE_TO_COLUMN = {
    "balance": "ОФП ФНС",
    "pnl": "ОФР ФНС",
    "equity": "ОИК ФНС",
    "cashflow": "ОДДС ФНС",
}


def save_sheet_result_as_html(sr: SheetResult, out_path: Path):
    title_block = f"<h2>{html.escape(sr.title)}</h2>" if sr.title else ""
    date_block = f"<p>{html.escape(sr.date)}</p>" if sr.date else ""

    html_doc = f"""<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="utf-8">
    <title>{html.escape(sr.title or sr.sheet_name)}</title>
    <style>
        body {{ font-family: Arial, sans-serif; font-size: 12px; padding: 20px; }}
        table {{ border-collapse: collapse; }}
        th {{ background: #e8e8e8; padding: 4px 8px; }}
        td {{ padding: 3px 8px; }}
    </style>
</head>
<body>
{title_block}
{date_block}
{sr.html}
</body>
</html>
"""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(html_doc)

In [13]:
def build_target_folder_name(row: pd.Series) -> str:
    '''
    Формирование имени папки
    '''
    company_name = sanitize_filename(row["company_name"])
    doc_type = "РСБУ"
    period = sanitize_filename(row["Отчетный период"])
    company_id = sanitize_filename(row["company_id"])
    file_id = row["FileID"]

    #return f"[{company_name}]-[РСБУ]-[{period}]-[{company_id}]"
    return f'[{company_name}]-[{doc_type}]-[{period}]-[{company_id}]-[{file_id}]'


def process_registry_row(row: pd.Series, row_idx: int) -> dict:
    '''
    Обработка одной строки реестра
    '''
    result_flags = {
        "ОФП ФНС": "",
        "ОФР ФНС": "",
        "ОИК ФНС": "",
        "ОДДС ФНС": "",
    }

    period = int(row["Отчетный период"])
    folder_name = build_target_folder_name(row)
    target_dir = BASE_SAVE_DIR / folder_name
    temp_dir = target_dir / "_tmp"

    try:
        orgid = get_fns_org_id_from_companies(row, companies_df)
        if orgid is None:
            print_red(f"[{row_idx}] ORG_ID / Код ФНС не найден")
            return result_flags
        
        target_dir.mkdir(parents=True, exist_ok=True)
        if temp_dir.exists():
            shutil.rmtree(temp_dir, ignore_errors=True)
        temp_dir.mkdir(parents=True, exist_ok=True)

        detail_id = get_detail_id_for_period(orgid, period, session)
        if detail_id is None:
            print_red(f"[{row_idx}] Не найден detail_id для ORG_ID={orgid}, period={period}")
            return result_flags

        zip_path = temp_dir / f"bfo_{orgid}_{period}_{detail_id}.zip"
        extract_dir = temp_dir / "unzipped"

        download_bfo_zip(orgid, period, detail_id, zip_path, session)
        extract_zip(zip_path, extract_dir)

        xlsx_path = find_first_xlsx(extract_dir)
        if xlsx_path is None:
            print_red(f"[{row_idx}] XLSX не найден после распаковки для ORG_ID={orgid}, period={period}")
            return result_flags

        file_result = process_file(str(xlsx_path))

        for sr in file_result.sheets:
            if sr.html is None:
                continue
            if sr.report_type not in REPORTTYPE_TO_FILENAME:
                continue

            html_name = REPORTTYPE_TO_FILENAME[sr.report_type]
            html_path = target_dir / html_name
            save_sheet_result_as_html(sr, html_path)

            result_col = REPORTTYPE_TO_COLUMN[sr.report_type]
            result_flags[result_col] = "V"

        return result_flags

    except Exception as exc:
        print_red(f"[{row_idx}] Ошибка: {exc}")
        return result_flags

    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)

In [11]:
import pandas as pd
from typing import Optional

def get_rep_versions_for_row_pos(df: pd.DataFrame, pos: int) -> Optional[int]:
    """
    Возвращает позицию строки с номером pos в группе строк, у которых совпадают (Тип документа, Отчетный период, ИНН), при сортировке по FileID по убыванию.
    pos — порядковый номер строки (0-based).
    """
    if pos < 0 or pos >= len(df):
        return None

    row = df.iloc[pos]

    doc_type = row['Тип документа']
    period = row['Отчетный период']
    company_name = row['company_name']

    mask = (
        (df['Тип документа'] == doc_type) &
        (df['Отчетный период'] == period) &
        (df['company_name'] == company_name)
    )

    group = df.loc[mask].copy()

    if group.empty:
        return None

    group['FileID__num'] = group['FileID'].astype(float)
    group_sorted = group.sort_values('FileID__num', ascending=False)
    
    orig_idx = df.index[pos]
    sorted_index = list(group_sorted.index)
    rank = sorted_index.index(orig_idx)
    
    return len(sorted_index)

In [14]:
df =  pd.read_excel(INPUT_XLSX, dtype = {'Тип документа': str, 'Отчетный период': str, 'company_name': str, 'ИНН': str, 'Ссылка на карточку': str, 'Последняя активность': str,'FileID': str})

# Отбор нужных строк: где Тип документа == Годовая бухгалтерская отчетность (все формы) и год "['2021','2022','2023', '2024', '2025']"
target_years = ['2021','2022','2023', '2024', '2025']
target_rep_type = 'Годовая бухгалтерская отчетность (все формы)'

#print(df["Тип документа"].astype(str).str.strip().eq(target_rep_type))
#print(df["Отчетный период"].astype(str).str.strip().isin(target_years))

mask = (df["Тип документа"].astype(str).str.strip().eq(target_rep_type) & df["Отчетный период"].astype(str).str.strip().isin(target_years))
target_indices = df.index[mask][:MAX_ROWS_TO_PROCESS].tolist()

print(f"Найдено строк для обработки: {len(target_indices)}")

for i in target_indices:
    row = df.loc[i]

    print(
        f"Обработка строки index={i}, "
        f"FileID={row['FileID']}, "
        f"company_name={row['company_name']}, "
        f"period={row['Отчетный период']}, "
        f"ИНН={row.get('ИНН')}"
    )

    num_vers = get_rep_versions_for_row_pos(df, i)

    if num_vers > 1:
        print(f"Строка #{i}: для заданной компании и отчетного периода найдено {num_vers} версий отчетности. Пропускаем загрузку с сайта ФНС")
    else:
        flags = process_registry_row(row, i)
        for col, value in flags.items():
            df.at[i, col] = value

df.to_excel(OUTPUT_XLSX, index=False)
print(f"Завершено. Результат сохранен в: {OUTPUT_XLSX}")

Найдено строк для обработки: 786
Обработка строки index=0, FileID=1920243, company_name=ООО "А Спэйс", period=2025, ИНН=9724021393


c:\Users\GaV\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Обработка строки index=4, FileID=1872978, company_name=ООО "А Спэйс", period=2024, ИНН=9724021393


c:\Users\GaV\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Обработка строки index=8, FileID=1830994, company_name=ООО "А Спэйс", period=2023, ИНН=9724021393


...

Обработка строки index=3768, FileID=1841881, company_name=ООО "Кеарли Групп", period=2023, ИНН=9715397054


c:\Users\GaV\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Обработка строки index=3769, FileID=1841875, company_name=ООО "Кеарли Групп", period=2022, ИНН=9715397054


c:\Users\GaV\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Завершено. Результат сохранен в: C:\Users\GaV\Desktop\Parsers\output_list_files.xlsx


In [ ]:
import pandas as pd
from pathlib import Path
from pypdf import PdfReader


def fill_pdf_page_counts():
    df = pd.read_excel(OUTPUT_XLSX)

    if 'Число страниц' not in df.columns:
        df['Число страниц'] = pd.NA

    for i, row in df.iterrows():
        file_path = row.get('Путь к файлу')

        if pd.isna(file_path):
            continue

        file_path = str(file_path).strip()
        if not file_path or file_path == '[Error]':
            continue

        path = Path(file_path)

        if not path.exists() or not path.is_file():
            continue

        if path.suffix.lower() != '.pdf':
            continue

        try:
            reader = PdfReader(str(path))
            df.loc[i, 'Число страниц'] = len(reader.pages)
            print(f"{i}: {path.name} -> {len(reader.pages)}")
        except Exception:
            continue

    df.to_excel(OUTPUT_XLSX, index=False)


fill_pdf_page_counts()